# 05 · RAG as a Tool

In notebook 03 we searched the documents on **every** question. But "What is 2 + 2?" does not need the shop's policies, and "Where is order A1002?" needs the order system, not the documents.

Better: make the document search **a tool** (Module 02) and let the agent **decide** when to use it. The agent now has two tools:

| Tool | Gives the agent |
|---|---|
| `search_knowledge_base` | The shop's policies (RAG) |
| `lookup_order` | A customer's order details (private data) |

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("Ready")

Ready


In [2]:
import re
from pathlib import Path

def load_chunks():
    """Read every .txt file in knowledge_base/ and split it into paragraphs. Each paragraph is a 'chunk'."""
    chunks = []
    for file in sorted(Path("knowledge_base").glob("*.txt")):
        text = file.read_text(encoding="utf-8")
        for paragraph in re.split(r"\n\s*\n", text):    # a blank line separates paragraphs
            if paragraph.strip():
                chunks.append({"source": file.name, "text": paragraph.strip()})
    return chunks

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks from the knowledge base")

Loaded 15 chunks from the knowledge base


In [3]:
def embed(texts):
    """Turn a list of texts into a list of vectors (a vector is just a long list of numbers)."""
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in response.data]

def similarity(vector_a, vector_b):
    """How close in meaning are two texts? Higher = closer.
    OpenAI vectors all have length 1, so multiplying them pair by pair and adding up is enough."""
    return sum(a * b for a, b in zip(vector_a, vector_b))

In [4]:
# Embed every chunk ONCE and keep the vectors. We reuse them for every question.
chunk_vectors = embed([chunk["text"] for chunk in chunks])

def retrieve(question, top_k=3):
    """Find the top_k chunks whose meaning is closest to the question."""
    question_vector = embed([question])[0]
    scored = [(similarity(question_vector, vector), chunk) for vector, chunk in zip(chunk_vectors, chunks)]
    scored.sort(key=lambda pair: pair[0], reverse=True)    # best match first
    return scored[:top_k]

## The two tools
The order data is made up, as in Module 02. Dates are worked out from today so the example always makes sense.

In [5]:
from datetime import date, timedelta

def days_ago(n):
    return (date.today() - timedelta(days=n)).isoformat()

ORDERS = {
    "A1001": {"item": "Wireless headphones", "price": 79.00, "status": "delivered", "delivered_on": days_ago(18)},
    "A1003": {"item": "Coffee grinder", "price": 45.00, "status": "delivered", "delivered_on": days_ago(80)},
}

def lookup_order(order_id):
    """Order details. We add 'days since delivery' so the model does not have to do date maths."""
    order = ORDERS.get(order_id.strip().upper())
    if order is None:
        return f"No order found with id {order_id}."
    details = dict(order)
    if order["status"] == "delivered":
        details["days_since_delivery"] = (date.today() - date.fromisoformat(order["delivered_on"])).days
    return json.dumps(details)

def search_knowledge_base(question):
    """Find the shop policy text that best matches the question."""
    results = retrieve(question, top_k=5)
    return "\n".join(f"[{chunk['source']}] {chunk['text']}" for score, chunk in results)

## Describe them to the model
The descriptions tell the agent **when** each tool is useful, so write them carefully.

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search the shop's policies (refunds, shipping, warranty, payments). Use it for any question about how the shop works. Ask about one topic at a time and search again for another topic.",
            "parameters": {
                "type": "object",
                "properties": {"question": {"type": "string", "description": "What to look up, in plain words"}},
                "required": ["question"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Get the details of one customer order: item, price, status and delivery date.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string", "description": "The order id, for example A1001"}},
                "required": ["order_id"],
            },
        },
    },
]

available_tools = {"search_knowledge_base": search_knowledge_base, "lookup_order": lookup_order}

def run_tool(name, args):
    if name not in available_tools:
        return f"Unknown tool: {name}"
    try:
        return str(available_tools[name](**args))
    except Exception as error:
        return f"The tool failed: {error}"

## The agent loop
The same loop as Module 02: ask the model, run the tools it requests, repeat until it answers.

In [7]:
SYSTEM = """You are a friendly support assistant for an online shop.
Answer questions about how the shop works ONLY from search_knowledge_base results. If the answer is not there, say you don't know.
If a question has several parts, search for each part separately. Whenever the customer mentions an order id, use lookup_order for it.
Keep answers to 3 short sentences."""

def run_agent(question, max_steps=5):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        messages.append(message)

        if not message.tool_calls:          # no tool requested: this is the final answer
            return message.content

        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = run_tool(tool_call.function.name, args)
            print(f"  [step {step}] {tool_call.function.name}({args})")
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "Stopped: too many steps."

## Watch the agent choose
**A question about shop policy.** It should search the knowledge base.

In [8]:
print(run_agent("How long does express shipping take, and how much does it cost?"))

  [step 1] search_knowledge_base({'question': 'express shipping duration'})
  [step 1] search_knowledge_base({'question': 'express shipping cost'})


Express shipping takes 1 to 2 business days. The cost for express shipping is always $12.


**A question that needs both the order and the policy.** The agent must look up the order to learn when it was delivered, and search the policies for the refund window and how refunds are paid.

In [9]:
print(run_agent("My order A1001 arrived a while ago. Can I still get a refund, and how long until the money arrives?"))

  [step 1] lookup_order({'order_id': 'A1001'})


  [step 1] search_knowledge_base({'question': 'refund policy'})


  [step 1] search_knowledge_base({'question': 'how long until refund arrives'})


You can still request a refund for your order A1001 since it was delivered within the last 30 days. The refund will be processed back to your original payment method, and the money typically arrives within 5 to 7 business days after we receive the returned item. If your item is faulty, you can return it without any shipping costs.


Same question, but for an order delivered 80 days ago:

In [10]:
print(run_agent("What about order A1003? Can I still get a refund for that one?"))

  [step 1] lookup_order({'order_id': 'A1003'})


  [step 2] search_knowledge_base({'question': 'refund policy'})


Unfortunately, you cannot get a refund for order A1003 as it was delivered over 30 days ago. Refund requests must be made within 30 days of delivery. However, if the coffee grinder is faulty, it may still be covered by the warranty.


**A question that needs no tool.**

In [11]:
print(run_agent("Hi there! Thanks for helping me earlier."))

You're welcome! How can I assist you today?


## Recap
* RAG works well **as a tool**. The agent decides when a document search is worth doing.
* Policies come from the **knowledge base**. Private data comes from **order tools**. One agent can use both.
* The model combines the results: order date from one tool, refund rules from the other.
* **The model chooses the search words.** A vague search like "refund policy" can rank the chunk about *how refunds are paid* only 5th, so we return 5 results and tell the model to search each part of a question separately. If an agent misses a detail, check what it searched for first.

In the real-life script, `support_assistant.py`, we add **long-term memory** too. The same shop assistant then has tools, a knowledge base **and** a memory of each customer.